# Benchmark: Fig. 2 of *Time-dependent Neural Galerkin Method for Quantum Dynamics*

Issue #24. A. Sinibaldi, D. Hendry, F. Vicentini, G. Carleo, arXiv:2412.11778.

**Physics.** Global quench of the 2D transverse-field Ising model on a periodic $L\times L$ lattice,
$$H = -J\sum_{\langle ij\rangle}\sigma^z_i\sigma^z_j - h\sum_i \sigma^x_i,\qquad J = 1,$$
from $|\Psi_0\rangle = |+\rangle^{\otimes N}$ to $h = 5$ (paramagnetic), $h = 3.044\approx h_c^{2D}$ (critical) and $h = 2$ (ferromagnetic). We track $\langle\sigma^x\rangle(t) = \frac1N\sum_i\langle\sigma^x_i\rangle$ and the normalized loss $\mathcal L(t)/N$,
$$\mathcal L(t) = \frac{\big\|P_\perp\big(\partial_t|\Psi\rangle + iH|\Psi\rangle\big)\big\|^2}{\langle\Psi|\Psi\rangle},\qquad P_\perp = 1 - \frac{|\Psi\rangle\langle\Psi|}{\langle\Psi|\Psi\rangle}.$$
$\mathcal L$ is zero on the exact solution and blind to any change of norm or global phase, so it measures only the physical error.

**Methods compared** (same observables, same loss):

| Method | Code | Ansatz |
| --- | --- | --- |
| t-NQG (the paper) | `QELM.NQS.dynamics.neural_galerkin` | $|\Psi(t)\rangle = |A_w\rangle + \sum_{i=1}^M c_i(t)|\phi_i\rangle$, $c_i(\tau)=\sum_k\gamma_{ik}(e^{i\omega_k\tau}-1)$, windows $\Delta T = 0.2$ |
| TDVP (our custom function) | `QELM.NQS.dynamics.run_tdvp` | one complex RBM started exactly at $|+\rangle^{\otimes N}$ |
| ED | `QELM.NQS.dynamics.exact_time_evolution` | exact, only for $N\le 16$ |

The TDVP loss is the same $\mathcal L(t)$, evaluated with $\partial_t\Psi = \sum_k\dot\theta_k\partial_k\Psi$ and the $\dot\theta$ that TDVP integrates.

**Paper settings** (6×6): $M = 18$ RBMs, $N_b = 128$ frequencies, $\Delta T = 0.2$, Simpson's rule with 257 points, 512 samples per integration point, Adam. The paper does not give the RBM density, the learning rate, the number of iterations, or the final time; see `documentation/neural_galerkin.md` for our choices and for the one change in the estimator (one set of samples from a time-independent mixture serves all integration points, an exact rewriting of $\mathcal L(t)$ as quadratic forms in $c(t)$).

**Unit and physics checks:** `tests/QELM/NQS/dynamics/test_neural_galerkin.py` and `test_galerkin_benchmark.py` (Hamiltonian vs Kronecker products, loss vs dense vectors, gauge invariance, $\mathcal L = 0$ on the exact solution, Monte Carlo vs exact sums, 3×3 quench vs ED, energy conservation).

In [ ]:
import numpy as np

from QELM.NQS.dynamics.galerkin_benchmark import (
    FIELDS,
    load_trajectories,
    main,
    plot_errors,
    plot_figure2,
)
from QELM.NQS.utils.outputs import find_repo_root

ROOT = find_repo_root()
DATA = ROOT / "data" / "benchmarks" / "neural_galerkin_2026"  # gitignored
PLOTS = ROOT / "outputs" / "neural_galerkin_2026"
RUN = False  # True: recompute (about 2 h for 4x4 and 5 h for 6x6 on 4 CPU cores)

## Runs

The command line does the same: `uv run python -m QELM.NQS.dynamics.galerkin_benchmark --L 6 --T 0.6 --out data/benchmarks/neural_galerkin_2026/L6`. Results are cached per field in `h=<h>.npz`; a run skips what is already there.

In [ ]:
if RUN:
    main(["--L", "4", "--T", "1.0", "--out", str(DATA / "L4")])
    main(
        [
            "--L",
            "6",
            "--T",
            "0.6",
            "--n-iter",
            "200",
            "--n-samples",
            "1024",
            "--out",
            str(DATA / "L6"),
        ]
    )

## 4×4: validation against exact diagonalization

In [ ]:
def load(L: int) -> dict:
    out = {}
    for h in FIELDS:
        path = DATA / f"L{L}" / f"h={h:g}.npz"
        if path.exists():
            out[h] = load_trajectories(path)
    return out


res4 = load(4)
for h, trs in res4.items():
    ed = trs["ED"]
    for key in ("tNQG", "TDVP"):
        if key in trs:
            tr = trs[key]
            err = np.abs(tr.x - np.interp(tr.times, ed.times, ed.x))
            print(
                f"h={h:<6g} {tr.method:<14} max|dx|={err.max():.1e} "
                f"mean|dx|={err.mean():.1e} median L/N={np.median(tr.loss / 16):.1e} "
                f"wall={tr.wall_time:.0f}s"
            )
if res4:
    plot_figure2(res4, 16, PLOTS / "figure2_4x4.png", title="4x4")
    plot_errors(res4, PLOTS / "errors_4x4.png", 16)

## 6×6: the upper panels of Fig. 2

No exact reference here ($2^{36}$ amplitudes); compare with Fig. 2 of the paper.

In [ ]:
res6 = load(6)
if res6:
    print(plot_figure2(res6, 36, PLOTS / "figure2_6x6.png", title="6x6"))

## Fig. 4: extrapolation beyond the training interval

$h = 3.044$, $M = 6$, one window trained on $[0, 2]$ and evaluated up to $t = 3$ (light green: training interval). This is the expensive part; run it on a cluster:

`uv run python -m QELM.NQS.dynamics.galerkin_benchmark --L 6 --T 2.0 --window 2.0 --extrapolate 1.0 --n-basis 6 --fields 3.044 --methods tNQG --out data/benchmarks/neural_galerkin_2026/fig4`

In [ ]:
fig4 = DATA / "fig4" / "h=3.044.npz"
if fig4.exists():
    plot_figure2(
        {3.044: load_trajectories(fig4)}, 36, PLOTS / "figure4.png", train_until=2.0
    )